# Task A Run 20 -- translate to English, then use an English hate model

The organisers' own published method on HASTIKA (`shankarb14/SLM-Impact`) normalises the
text before classifying it:

```
romanized comment -> IndicXlit -> Kannada script -> IndicTrans2 -> English -> encoder
```

This notebook tests the **normalisation half** of that idea, not the model half. Their
encoders are frozen and their paper is an efficiency study, which is why it uses TinyBERT and
MobileBERT; a frozen encoder was already measured here at 0.71. What is worth borrowing is
translating the input, then fine-tuning an English hate-speech model on it.

## Why this is worth a run

MuRIL's tokenizer destroys exactly the words that carry the signal: `sule` becomes `su` +
`##le`, `soole` becomes `so` + `##ole`, and 74.4% of word types appear once. Translation
removes that problem at its root rather than working around it, and `GroNLP/hateBERT` is
pretrained on English hate speech, a domain match nothing in the current stack has.

It would also be the most **decorrelated** ensemble member available. Run 10 showed the
existing members failing on the same 16.4% of rows; a completely different input
representation is the thing most likely to break that.

## The risk, and why stage 1 exists

Machine translation routinely sanitises or mangles profanity, and 18% of Task A's rows carry
a slur the model already gets right 91.6% of the time. If the slurs do not survive
translation, the signal is gone and no downstream model can recover it.

So stage 1 translates a sample and measures slur survival directly, in about ten minutes.
**If that gate fails, the notebook stops itself** rather than spending the remaining two
hours.

| stage | what | time |
|---|---|---|
| 1 | translate 300 slur-bearing comments, measure survival, print examples | ~10 min |
| 2 | translate the whole corpus, cached | ~45 min |
| 3 | fine-tune hateBERT on the translated text, five-fold | ~60 min |
| 4 | disagreement with the char n-gram SVM, and a blend check | ~2 min |

Set **Accelerator** to `GPU T4 x2` or `GPU P100` and **Internet** on, then
**Save Version -> Save & Run All**.

In [ ]:
import json, os, pathlib, shutil, subprocess, sys, zipfile

WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
print("repo:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)

import numpy as np
import pandas as pd
import torch
from hastika.common.preprocessing import dedupe_index

assert torch.cuda.is_available(), "no GPU -- select a CUDA-enabled runtime"
print("gpu:", torch.cuda.get_device_name(0))
train = pd.read_csv("data/raw/binary_train.csv")
keep = dedupe_index(train["Comment"].tolist(), train["Label"].tolist(), "task A")
print(f"raw labelled rows: {len(train)}; deduplicated rows used for fitting: {len(keep)}")
assert len(keep) == 6401, len(keep)

def run(cmd, log=None):
    print("$", " ".join(cmd), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Build the normaliser

Word-level language ID first, as the organisers do: only tokens judged to be romanized
Kannada are transliterated. Handles, hashtags, URLs, numerals, emoji and recognised English
words are masked and restored afterwards, so translation does not mangle them.

The distilled 200M IndicTrans2 is used rather than the 1B: several times faster, and the
quality difference on short social-media sentences is small.

In [ ]:
subprocess.run("pip install -q ai4bharat-transliteration IndicTransToolkit nltk", shell=True)
import re
import nltk
nltk.download("words", quiet=True)
from nltk.corpus import words as nltk_words

EN_VOCAB = {w.lower() for w in nltk_words.words()}
NEUTRAL = re.compile(r"^(?:@\w+|#\w+|https?://\S+|[\d.,:%/+-]+|[^\w\s]+)$")
KANNADA = re.compile(r"[\u0C80-\u0CFF]")

def tag(tok):
    if NEUTRAL.match(tok) or KANNADA.search(tok):
        return "neutral"
    return "en" if tok.lower() in EN_VOCAB else "kn"

def mask(text):
    kept, pieces = {}, []
    for i, tok in enumerate(text.split()):
        if tag(tok) == "kn":
            pieces.append(tok)
        else:
            ph = f"@@{i}@@"
            kept[ph] = tok
            pieces.append(ph)
    return " ".join(pieces), kept

def unmask(text, kept):
    for ph, tok in kept.items():
        text = text.replace(ph, tok)
    return text

from ai4bharat.transliteration import XlitEngine
xlit = XlitEngine("kn", beam_width=4, rescore=False)

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
try:
    from IndicTransToolkit import IndicProcessor
except ImportError:
    try:
        from IndicTransToolkit.IndicTransToolkit import IndicProcessor
    except ImportError:
        from IndicTransToolkit.processor import IndicProcessor
NMT = "ai4bharat/indictrans2-indic-en-dist-200M"
nmt_tok = AutoTokenizer.from_pretrained(NMT, trust_remote_code=True)
nmt = AutoModelForSeq2SeqLM.from_pretrained(NMT, trust_remote_code=True).to("cuda").eval()
ip = IndicProcessor(inference=True)

def to_kannada(text):
    masked, kept = mask(text)
    if not masked.strip():
        return unmask(masked, kept)
    out = xlit.translit_sentence(masked, lang_code="kn")
    if isinstance(out, dict):
        out = out.get("kn") or next(iter(out.values()))
    if isinstance(out, (list, tuple)):
        out = out[0]
    return unmask(str(out), kept)

probe = to_kannada("nin sule maga")
print("transliteration probe:", repr(probe))
assert any("\u0C80" <= ch <= "\u0CFF" for ch in probe), "IndicXlit returned no Kannada script"

def to_english(sentences, bs=32):
    done = []
    for i in range(0, len(sentences), bs):
        chunk = ip.preprocess_batch(sentences[i:i + bs], src_lang="kan_Knda", tgt_lang="eng_Latn")
        enc = nmt_tok(chunk, truncation=True, padding=True, max_length=256,
                      return_tensors="pt").to("cuda")
        with torch.no_grad():
            gen = nmt.generate(**enc, num_beams=5, max_length=256)
        done += ip.postprocess_batch(nmt_tok.batch_decode(gen, skip_special_tokens=True),
                                     lang="eng_Latn")
    return done
print("normaliser ready")

## 2. Stage 1: does the signal survive translation?

Specific rather than impressionistic. Take comments containing one of the slurs measured as
most predictive on this corpus, translate them, and ask whether anything abusive comes out
the other side. Ten are printed in full, so a failure mode is visible rather than inferred.

In [ ]:
from hastika.common.preprocessing import clean, dedupe_index

KN_SLUR = re.compile(r"\b(thu+|sule|soole|sulle|dagar|magane|maklu|makla|bevarsi|thika|"
                     r"gandu|munde|nayi|randi|kachada|janmakke|boli|bosudi)\w*", re.I)
EN_ABUSE = re.compile(r"\b(fuck\w*|shit\w*|bitch\w*|bastard\w*|whore|slut|prostitut\w*|"
                      r"idiot\w*|stupid|nonsense|rascal|scoundrel|dog|bloody|damn|"
                      r"disgust\w*|shame\w*|filth\w*|useless|nasty|abuse\w*)\b", re.I)

tr = pd.read_csv("data/raw/binary_train.csv")
tr = tr.iloc[dedupe_index(tr["Comment"].tolist(), tr["Label"].tolist())].reset_index(drop=True)
tr["clean"] = [clean(t) for t in tr["Comment"]]
slur = tr[tr["clean"].str.contains(KN_SLUR)].sample(300, random_state=0)

kn = [to_kannada(t) for t in slur["clean"]]
en = to_english(kn)
survived = np.array([bool(EN_ABUSE.search(e)) for e in en])
empty = np.array([len(e.strip()) < 5 for e in en])
print(f"slur-bearing comments sampled: {len(en)}")
print(f"  translations carrying an English abusive term: {survived.mean():.1%}")
print(f"  translations empty or near-empty: {empty.mean():.1%}")
print()
for i in range(10):
    print("  romanized :", slur["clean"].iloc[i][:90])
    print("  kannada   :", kn[i][:90])
    print("  english   :", en[i][:90])
    print()

## 3. The gate

A pass does not mean the pipeline will win, only that it is worth measuring. A fail means the
abuse is being laundered out, so the run stops itself.

The threshold is deliberately lenient: the English lexicon is small and misses paraphrases,
so it undercounts survival. Below 25% is a clear failure.

In [ ]:
GATE = 0.25
print(f"survival {survived.mean():.1%} against a {GATE:.0%} gate; empty {empty.mean():.1%}")
if survived.mean() < GATE or empty.mean() > 0.2:
    raise SystemExit("GATE FAILED: translation is removing the signal. Stop and record it.")
print("gate passed, continuing")

## 4. Stage 2: translate the whole corpus

Training rows and validation inputs, cached. The translations then replace the `Comment`
column in the session's copy of the released files, so the tuned recipe in
`hastika.models.muril` runs unchanged on English text.

In [ ]:
va = pd.read_csv("data/raw/binary_validation_inputs.csv")
va["clean"] = [clean(t) for t in va["Comment"]]
CACHE = pathlib.Path("artifacts/data"); CACHE.mkdir(parents=True, exist_ok=True)

for name, df in [("train", tr), ("val", va)]:
    path = CACHE / f"english_{name}.csv"
    if path.exists():
        print("reusing", path)
        continue
    kn_all = [to_kannada(t) for t in df["clean"]]
    en_all = to_english(kn_all)
    out = df[["id"]].copy()
    out["Comment"] = en_all
    if "Label" in df:
        out["Label"] = df["Label"].values
    out.to_csv(path, index=False)
    print("wrote", path, len(out))

eng_tr = pd.read_csv(CACHE / "english_train.csv")
eng_va = pd.read_csv(CACHE / "english_val.csv")
eng_tr.to_csv("data/raw/binary_train.csv", index=False)
eng_va[["id", "Comment"]].to_csv("data/raw/binary_validation_inputs.csv", index=False)
print(eng_tr.shape, eng_va.shape)
print(eng_tr["Comment"].head(3).tolist())

## 5. Stage 3: fine-tune an English hate model on the translated text

`GroNLP/hateBERT` is BERT continued on English abusive language, the closest pretrained match
to the translated text. Five folds on the same split seed as every other Task A experiment,
so the number compares directly with the 0.8073 TF-IDF floor and the 0.8128 TAPT MuRIL.

In [ ]:
TAG = "a_english_hatebert"
run([sys.executable, "-u", "-m", "hastika.models.muril", "--tag", TAG,
     "--model", "GroNLP/hateBERT", "--folds", "5", "--seeds", "42", "--epochs", "6",
     "--bs", "16", "--eval-bs", "64", "--select", "last", "--reinit-layers", "2",
     "--no-demojize"], log=f"artifacts/logs/{TAG}.log")

## 6. Stage 4: is it a useful ensemble member?

Its own score matters less than whether it fails on different rows. Run 10 showed the
opposite case: a very different model that disagreed on 23% of rows and still added nothing,
because it was the one that was wrong on them. `either right` is the ceiling a perfect blend
of the pair could reach.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC

y = (eng_tr["Label"] == "Hate").astype(int).values
eng_oof = np.load(pathlib.Path("artifacts/runs") / TAG / "oof_probs.npy")
Xk = np.array([clean(t, demojize=True) for t in tr["Comment"]])
svm_oof = np.zeros((len(y), 2))
for a, b in StratifiedKFold(5, shuffle=True, random_state=42).split(Xk, y):
    m = make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2,
                                      sublinear_tf=True),
                      CalibratedClassifierCV(LinearSVC(C=0.5, class_weight="balanced"), cv=3))
    svm_oof[b] = m.fit(Xk[a], y[a]).predict_proba(Xk[b])

se = f1_score(y, eng_oof.argmax(1), average="macro")
ss = f1_score(y, svm_oof.argmax(1), average="macro")
print(f"  hateBERT on translated English : {se:.4f}")
print(f"  char n-gram SVM on romanized   : {ss:.4f}")
print("  references: TF-IDF floor 0.8073, TAPT MuRIL 0.8128")
dis = (eng_oof.argmax(1) != svm_oof.argmax(1)).mean()
either = ((eng_oof.argmax(1) == y) | (svm_oof.argmax(1) == y)).mean()
print(f"\n  they disagree on {dis:.3f} of rows; either is right on {either:.3f}")
best = max(((w, f1_score(y, ((w * svm_oof[:, 1] + (1 - w) * eng_oof[:, 1]) > 0.5).astype(int),
                         average="macro")) for w in np.arange(0, 1.01, 0.05)), key=lambda t: t[1])
print(f"  best in-sample blend: w_svm={best[0]:.2f} -> {best[1]:.4f} (optimistic)")

## 7. Preserve outputs

The translated CSVs are the expensive part and the reusable one. Keep them even if the model
loses: any later idea on translated text starts from them for free.

In [ ]:
OUT = pathlib.Path("/kaggle/working/task_a_english_outputs")
OUT.mkdir(parents=True, exist_ok=True)
for f in CACHE.glob("english_*.csv"):
    shutil.copy2(f, OUT / f.name)
for f in pathlib.Path("artifacts/logs").glob("*.log"):
    shutil.copy2(f, OUT / f.name)
p = pathlib.Path("artifacts/runs") / TAG / "oof_probs.npy"
if p.exists():
    shutil.copy2(p, OUT / "english_oof_probs.npy")
np.save(OUT / "svm_oof_probs.npy", svm_oof)
json.dump({"slur_survival": float(survived.mean()), "english_oof": float(se),
           "svm_oof": float(ss), "disagreement": float(dis), "either_right": float(either)},
          open(OUT / "result.json", "w"), indent=2)
print(sorted(x.name for x in OUT.iterdir()))

## 8. What to do with the result

Record the slur-survival rate and both out-of-fold scores in `docs/EXPERIMENTS.md`,
including a gate failure. A negative result closes off the organisers' own normalisation
pipeline for this use, which is otherwise an obvious thing to keep reaching for.

- **Gate fails.** Translation launders the abuse. Record it and stop.
- **Below 0.80, but disagreement above 0.20 and `either right` well above both.** Not a
  standalone model, but a candidate third ensemble member.
- **Near or above 0.8128.** A genuinely different representation that works, attacking the
  tokenizer problem at its root rather than around it.

This notebook overwrites `data/raw/binary_train.csv` and `binary_validation_inputs.csv` in
the Kaggle clone with their translated versions. That is local to the session and the repo is
untouched, but do not run other Task A work in the same session afterwards.